# trellis.cpp on Kaggle — GGML/C++ TRELLIS.2 (no ComfyUI, no PyTorch at inference)

**Status: build + weight download both confirmed working on this exact Kaggle T4 image.**
All 14 CMake targets compiled clean (`trellis-cli`, `trellis-server`, smoke tests), and the
16.5GB GGUF weight set from `ilintar/trellis2-gguf` downloaded successfully.

**Also confirmed:** the CLI's image loader (`stb_image`) can't read interlaced/Adam7 PNGs —
a common silent export setting in some tools — even though other loaders (PIL, ComfyUI) read
the same file fine. Cell 10 now sanitizes the image with PIL before handing it to the CLI.

**Settings:** Accelerator = GPU T4 x2, Internet = ON.


## 1. Hard GPU + CUDA toolkit check

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError("No GPU detected. Session options -> Accelerator -> GPU T4 x2 -> save -> re-run.")
print(r.stdout)

nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True)
if nvcc.returncode != 0:
    raise RuntimeError("nvcc not found - CUDA toolkit missing.")
print(nvcc.stdout)


## 2. Build dependencies

In [ ]:
!apt-get update -qq
!apt-get install -y -qq cmake build-essential git > /dev/null
!cmake --version
!g++ --version


## 3. Clone (with submodules — vendored ggml/stb/xatlas live behind .gitmodules)

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/pwilkin/trellis.cpp.git
%cd /kaggle/working/trellis.cpp
!ls


## 4. Configure with CUDA — T4 is Turing (compute capability 7.5)
Pinning `CMAKE_CUDA_ARCHITECTURES=75` avoids nvcc guessing wrong or building for every arch.

**Confirmed fix for the `CUDA::cuda_driver` not found error:** on this Kaggle image the real
driver lib lives at a Julia artifacts path (`/root/.julia/artifacts/.../libcuda.so.1`), not
anywhere `FindCUDAToolkit` looks by default. Cell below auto-finds it, symlinks it in, and
does a clean reconfigure. Confirmed working end-to-end last run.

In [ ]:
import subprocess

candidates = subprocess.run(
    ["bash", "-c", "find / -xdev -name 'libcuda.so*' 2>/dev/null"],
    capture_output=True, text=True
).stdout.strip().splitlines()
print("Found candidates:")
for c in candidates:
    print(" ", c)
assert candidates, "No libcuda.so* found anywhere on this box."

real_libs = [c for c in candidates if ".so.1" in c]
source_lib = real_libs[0] if real_libs else candidates[0]
print("Using as source:", source_lib)

target = "/usr/local/cuda/lib64/libcuda.so"
subprocess.run(["ln", "-sf", source_lib, target], check=True)
subprocess.run(["ldconfig"], check=True)
print(subprocess.run(["ls", "-la", target], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/trellis.cpp
!rm -rf build
!cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DCMAKE_BUILD_TYPE=Release -DCUDAToolkit_ROOT=/usr/local/cuda


### Fallback — only run this if the cell above still errors on `CUDA::cuda_driver`
Passes the found library path directly to CMake's own cache variable, bypassing auto-detection.

## 5. Build (confirmed working — full compile takes a while, be patient)

In [ ]:
import multiprocessing
n = multiprocessing.cpu_count()
%cd /kaggle/working/trellis.cpp
!cmake --build build -j{n}


## 6. Confirm the binary + real CLI usage

In [ ]:
!find /kaggle/working/trellis.cpp/build -maxdepth 3 -iname 'trellis*' -type f
print('--- --help output ---')
!/kaggle/working/trellis.cpp/build/trellis-cli --help || true


## 7. HF_TOKEN (Kaggle Secrets) — optional (weights are ungated, download already worked without it)

In [ ]:
import os
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("No HF_TOKEN secret found (fine - weights are ungated):", e)

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF_TOKEN set.")
else:
    print("Continuing unauthenticated.")


## 8. Download the GGUF weights (confirmed working — 16.5GB, 12 files)

In [ ]:
!pip install -q huggingface_hub
from huggingface_hub import snapshot_download

MODELS_DIR = "/kaggle/working/trellis.cpp/models/trellis2"
snapshot_download(
    repo_id="ilintar/trellis2-gguf",
    local_dir=MODELS_DIR,
)
!find {MODELS_DIR} -maxdepth 2


## 9. Final step.
Save packages as input for a reusable notebook

In [ ]:
# Package artifacts for reuse, then slim under the 20GB output cap
%cd /kaggle/working
!mkdir -p dist

# The binary
!cp trellis.cpp/build/trellis-cli dist/

# Any shared libs built inside the repo that the binary links against
# (ggml often builds as libggml*.so rather than static)
!ldd trellis.cpp/build/trellis-cli | grep '/kaggle/working/trellis.cpp' | awk '{print $3}' | xargs -r -I{} cp -v {} dist/

# The weights (mv, not cp - no room for two copies)
!mv trellis.cpp/models/trellis2 dist/models

# Pre-download pip wheels so the runtime notebook installs offline-fast and version-locked
import numpy
!pip download -q -d dist/wheels rembg onnxruntime numpy=={numpy.__version__}

# Delete everything else so the committed output is just dist/
!rm -rf trellis.cpp
!du -sh dist dist/models
!ls -la dist